# Weather Data Analysis — Indian Cities
**Mumbai · Delhi · Dehradun · Jodhpur | 2000–2024**

This is a Python-based data analysis project that looks at daily weather patterns across four Indian cities over the last 25 years. The data is taken from NOAA's Global Summary of the Day (GSOD) database, which records daily weather observations from stations all around the world.

The notebook is split into:
- **Part 1** — Getting the Data
- **Part 2** — Cleaning the Data


---
# Part 1 — Getting the Data

The data is downloaded directly from NOAA's website using Python, so there is no manual step involved. Each city has a unique station ID, and for each year from 2000 to 2024, there is a separate CSV file at:

```
https://www.ncei.noaa.gov/data/global-summary-of-the-day/access/{year}/{station_id}.csv
```

All 25 files per city are downloaded and combined into one dataframe.

## 1.1 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import time
import random
import io
import warnings
from scipy import stats
from concurrent.futures import ThreadPoolExecutor, as_completed

warnings.filterwarnings('ignore')
print('imports done')

## 1.2 — Station IDs and Constants

The four station IDs below are the confirmed NOAA GSOD stations for each city. Jodhpur uses WMO station 42339 (ICAO: VIJO), which is a major IMD synoptic station with continuous records back to 2000.

In [ ]:
# station IDs from NOAA's ISD station history
CITIES = {
    'Mumbai':   {'station_id': '43003099999', 'lat': 19.07, 'lon': 72.87},
    'Delhi':    {'station_id': '42182099999', 'lat': 28.58, 'lon': 77.20},
    'Dehradun': {'station_id': '42189099999', 'lat': 30.32, 'lon': 78.03},
    'Jodhpur':  {'station_id': '42339099999', 'lat': 26.30, 'lon': 73.02},
}

START_YEAR = 2000
END_YEAR   = 2024

print('Cities and station IDs:')
for city, info in CITIES.items():
    print(f"  {city:10s}  station: {info['station_id']}  "
          f"lat: {info['lat']}  lon: {info['lon']}")

## 1.3 — Download

One problem encountered early was that NOAA's server kept returning HTTP 503 errors with the default `pd.read_csv(url)` method. This happens because NOAA's CDN treats plain Python requests as bots and blocks them.

The fix was to use a `requests.Session` with proper browser-like headers (a Chrome User-Agent, Accept-Language, and Referer), along with retry logic that waits and tries again if a 503 comes back. A small random delay between requests was also added so it does not look like automated scraping.

To speed things up, `ThreadPoolExecutor` is used to download 3 years simultaneously instead of one by one.

In [ ]:
# requests.Session is needed — plain requests.get() triggers
# 'URL has an invalid label' on NOAA's URL in some environments
# Session also lets us set headers once for all calls
session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Referer': 'https://www.ncei.noaa.gov/'
})


def download_one_year(station_id, year):
    """Download one year of GSOD data for one station.
    Returns (year, dataframe) on success, (year, None) on failure."""
    url = ('https://www.ncei.noaa.gov/data/global-summary-of-the-day'
           f'/access/{year}/{station_id}.csv')
    for attempt in range(4):  # retry up to 4 times
        try:
            time.sleep(random.uniform(0.4, 1.0))  # random delay looks human
            r = session.get(url, timeout=25)
            if r.status_code == 200 and len(r.content) > 200:
                return year, pd.read_csv(io.StringIO(r.text), low_memory=False)
            elif r.status_code in (503, 429):  # server busy, wait longer
                wait = (2 ** attempt) + random.uniform(0, 0.5)
                time.sleep(wait)
            else:
                break  # 404 or similar, no point retrying
        except Exception:
            time.sleep(2 ** attempt)
    return year, None


def download_city_data(city_name, station_id):
    """Download all years for one city using 3 parallel threads."""
    years  = list(range(START_YEAR, END_YEAR + 1))
    frames = {}
    with ThreadPoolExecutor(max_workers=3) as pool:
        futures = {pool.submit(download_one_year, station_id, yr): yr for yr in years}
        for future in as_completed(futures):
            year, df = future.result()
            if df is not None:
                frames[year] = df
                print(f'  downloaded {city_name} {year} — {len(df)} rows')
            else:
                print(f'  skipped    {city_name} {year} — no data')
    if not frames:
        print(f'  ERROR: no data at all for {city_name}')
        return None
    combined         = pd.concat([frames[y] for y in sorted(frames)], ignore_index=True)
    combined['CITY'] = city_name
    return combined


print('Download functions ready')

## 1.4 — Run Download for All Four Cities

In [ ]:
raw_data = {}
for city, info in CITIES.items():
    print(f"\n--- {city} (station {info['station_id']}) ---")
    raw_data[city] = download_city_data(city, info['station_id'])

print('\n' + '='*50)
print('DOWNLOAD SUMMARY')
print('='*50)
for city, df in raw_data.items():
    if df is not None:
        print(f'  {city:10s}: {len(df):,} rows, {df.shape[1]} columns')
    else:
        print(f'  {city:10s}: FAILED')

---
# Part 2 — Cleaning the Data

This is the most important part of the project. Raw GSOD data comes with several issues that need to be handled before any analysis can be done. Each cleaning step is in its own cell below, matching the README sections.

## 2.1 — Unit Conversion

NOAA stores everything in imperial units — temperatures in Fahrenheit, precipitation in inches, and wind speed in knots. All of these are converted to metric (°C, mm, and m/s) as the first step, since all analysis and comparisons are done in standard units.

In [ ]:
def fahrenheit_to_celsius(s): return (s - 32) * 5 / 9
def inches_to_mm(s):          return s * 25.4
def knots_to_ms(s):           return s * 0.514444
def miles_to_km(s):           return s * 1.60934

print('Unit conversion functions ready')
print('  Temperature : Fahrenheit  -> Celsius')
print('  Precipitation: inches     -> mm')
print('  Wind speed  : knots       -> m/s')
print('  Visibility  : miles       -> km')

## 2.2 — Missing Value Codes

NOAA does not use NaN for missing data. Instead it uses specific fill values like 9999.9 for temperature and 99.99 for precipitation. These had to be identified and replaced with actual NaN before doing anything else, because otherwise they appear as real observations and completely break the statistics.

In [ ]:
# these are NOAA's sentinel codes meaning 'no data recorded'
# they are NOT real values — 9999.9 is not a real temperature
MISSING_VALUES = {
    'TEMP': 9999.9,  # mean temperature
    'DEWP': 9999.9,  # dew point
    'SLP':  9999.9,  # sea level pressure
    'STP':  9999.9,  # station pressure
    'VISIB': 999.9,  # visibility
    'WDSP':  999.9,  # mean wind speed
    'MXSPD': 999.9,  # max sustained wind
    'GUST':  999.9,  # peak gust
    'MAX':  9999.9,  # max temperature
    'MIN':  9999.9,  # min temperature
    'PRCP':  99.99,  # precipitation
    'SNDP':  999.9,  # snow depth
}

print('Missing value sentinel map defined')
print(f'  {len(MISSING_VALUES)} variables covered')

## 2.3 — Duplicates

Some station records had more than one row for the same date. These are removed by keeping only the first occurrence.

## 2.4 — Impossible Values

After unit conversion, a physical plausibility check is applied to catch values that simply cannot exist in the real world.

**Important:** these bounds are absolute Earth-record limits, NOT city-level climate averages. A 48°C reading in Jodhpur during a heatwave is extreme but physically possible and stays. A 75°C reading is impossible anywhere on Earth and gets removed.

Checks applied:
- Temperature above 60°C or below -90°C → impossible
- Negative rainfall, wind speed, visibility → impossible
- Rainfall above 2000mm in a single day → impossible (world record is ~1825mm)
- MAX temperature lower than MIN on the same day → recording error
- Dew point higher than air temperature → physically impossible
- F→C overflow: values near NOAA's 9999.9°F sentinel that slipped through sentinel detection produce wildly large Celsius values — caught here

## 2.5 — The Outlier Problem

This is the most carefully designed part of the pipeline. A lot of standard data cleaning approaches flag statistical outliers and remove them. But in weather data, that is actually the wrong thing to do. A day with 900mm of rainfall is not a bad data point — it is the 2005 Mumbai floods. Removing it because it looks unusual would mean losing exactly the event this project is meant to study.

The pipeline separates unusual values into **three categories**, each treated differently:

| Category | What it is | What happens |
|----------|-----------|-------------|
| Tier 1 — Instrument errors | Physically impossible values | **Deleted** |
| Tier 2 — Real extreme events | Genuine severe weather, confirmed | **Kept + labelled** |
| Tier 3 — Statistical anomalies | Unusual but not confirmed | **Kept + scored** |

### Tier 2 — Real Extreme Events (Kept and Labelled)

These specific historical events are manually listed and protected. Any automated step that tries to flag them as errors is overridden.

In [ ]:
# IMD (India Meteorological Department) standard thresholds
IMD_HEAVY_RAIN   = 64.5    # mm/day — IMD 'Heavy Rain'
IMD_VERY_HEAVY   = 115.6   # mm/day — IMD 'Very Heavy Rain'
IMD_EXTREME_RAIN = 204.4   # mm/day — IMD 'Extremely Heavy Rain'
IMD_HEATWAVE     = 40.0    # degrees C — IMD heatwave threshold
IMD_SEVERE_HEAT  = 45.0    # degrees C — IMD severe heatwave

# known historical events — manually listed and permanently protected
# format: (city, year, month, variable, description)
KNOWN_EXTREMES = [
    ('Mumbai',   2005,  7, 'PRCP',
     'Mumbai 26 Jul 2005 floods — ~944mm in one day, 1000+ deaths, Mithi river overflow'),
    ('Mumbai',   2017,  8, 'PRCP',
     'Mumbai Aug 2017 monsoon floods — Sion 375mm in one day'),
    ('Delhi',    2002,  5, 'MAX',
     'Delhi May 2002 heatwave — above 47C, hundreds of deaths across north India'),
    ('Delhi',    2022,  5, 'MAX',
     'Delhi May 2022 heatwave — 49.2C at Mungeshpur, hottest in 122 years'),
    ('Dehradun', 2013,  6, 'PRCP',
     'Uttarakhand Jun 2013 cloudbursts — 5000+ deaths, Kedarnath disaster'),
    ('Jodhpur',  2019,  6, 'MAX',
     'Jodhpur Jun 2019 heatwave — part of wider Rajasthan 47C+ surge'),
]

print('Known extreme events registered:')
for ev in KNOWN_EXTREMES:
    print(f'  {ev[0]:10s}  {ev[1]}  Month {ev[2]:2d}  [{ev[3]}]')
    print(f'    {ev[4]}')

### Tier 3 — Statistical Anomalies (Kept and Scored)

Values that are not impossible and not a known extreme event, but are still unusually far from what is typical for that variable in that month.

**Why median + MAD instead of mean + standard deviation?**

If you use the regular mean and SD to compute a z-score, a real extreme event inflates the mean itself. This paradoxically makes the extreme event look *less* unusual than it actually is — the very thing you are trying to detect.

The median and MAD (Median Absolute Deviation) are not affected by extreme values in the same way because the median does not shift when one value is very large. This is called a robust estimator.

The formula used is the Iglewicz and Hoaglin (1993) modified z-score:

```
robust z = 0.6745 × (x − monthly median) / monthly MAD
```

Any value with `|robust z| > 3.5` is flagged as anomalous, but it stays in the dataset. Known extreme events that get flagged by this score have their flag overridden to False — because they are confirmed real events, not errors.

## 2.6 — Full Cleaning Function

The function below runs all steps above in sequence for one city's dataframe.

In [ ]:
def clean_city_data(df, city_name):
    df         = df.copy()
    rows_start = len(df)
    print(f'\nCleaning {city_name} — starting with {rows_start} rows')

    # ── parse date, extract year and month ──────────────────────────────
    # date is kept temporarily for duplicate detection, then dropped
    df['DATE'] = pd.to_datetime(df['DATE'], errors='coerce')
    bad_dates  = df['DATE'].isna().sum()
    df         = df.dropna(subset=['DATE']).copy()
    if bad_dates > 0:
        print(f'  dropped {bad_dates} rows with unreadable dates')
    df['YEAR']   = df['DATE'].dt.year
    df['MONTH']  = df['DATE'].dt.month
    df['SEASON'] = df['MONTH'].map({
        12: 'Winter',   1: 'Winter',   2: 'Winter',
        3:  'Pre-Monsoon', 4: 'Pre-Monsoon', 5: 'Pre-Monsoon',
        6:  'Monsoon',  7: 'Monsoon',  8: 'Monsoon',
        9:  'Post-Monsoon', 10: 'Post-Monsoon', 11: 'Post-Monsoon'
    })
    df = df.sort_values(['YEAR', 'MONTH', 'DATE']).reset_index(drop=True)

    # ── 2.2: replace NOAA sentinel fill values with NaN ─────────────────
    total_sentinels = 0
    for col, fill_val in MISSING_VALUES.items():
        if col in df.columns:
            mask              = df[col] == fill_val
            total_sentinels  += mask.sum()
            df.loc[mask, col] = np.nan
    print(f'  2.2 | replaced {total_sentinels} sentinel fill values with NaN')

    # ── 2.3: remove duplicate rows for the same date ────────────────────
    before = len(df)
    df     = df.drop_duplicates(subset=['DATE'], keep='first')
    print(f'  2.3 | removed {before - len(df)} duplicate rows')

    # ── 2.1: unit conversion (imperial -> metric) ────────────────────────
    for col in ['TEMP', 'DEWP', 'MAX', 'MIN']:
        if col in df.columns: df[col] = fahrenheit_to_celsius(df[col])
    for col in ['PRCP', 'SNDP']:
        if col in df.columns: df[col] = inches_to_mm(df[col])
    for col in ['WDSP', 'MXSPD', 'GUST']:
        if col in df.columns: df[col] = knots_to_ms(df[col])
    if 'VISIB' in df.columns:
        df['VISIB'] = miles_to_km(df['VISIB'])
    print('  2.1 | converted to metric (C, mm, m/s, km)')

    # ── 2.4: impossible negatives ────────────────────────────────────────
    neg_total = 0
    for col in ['PRCP', 'WDSP', 'MXSPD', 'GUST', 'SNDP', 'VISIB']:
        if col in df.columns:
            mask       = df[col] < 0
            neg_total += mask.sum()
            df.loc[mask, col] = np.nan
    if neg_total > 0:
        print(f'  2.4 | removed {neg_total} impossible negative values')

    # ── 2.4: physically impossible values (absolute Earth bounds) ────────
    impossible = 0
    for col in ['TEMP', 'MAX', 'MIN', 'DEWP']:      # world records: -89.2C / +56.7C
        if col in df.columns:
            bad = (df[col] < -90) | (df[col] > 60)
            impossible += bad.sum()
            df.loc[bad, col] = np.nan
    if 'PRCP' in df.columns:                         # world record 1-day: ~1825mm
        bad = df['PRCP'] > 2000
        impossible += bad.sum()
        df.loc[bad, 'PRCP'] = np.nan
    if 'WDSP' in df.columns:                         # world record sustained: ~113 m/s
        bad = df['WDSP'] > 90
        impossible += bad.sum()
        df.loc[bad, 'WDSP'] = np.nan
    if 'MAX' in df.columns and 'MIN' in df.columns:  # MAX must always be >= MIN
        bad = df['MAX'] < df['MIN']
        impossible += bad.sum()
        df.loc[bad, ['MAX', 'MIN']] = np.nan
    if 'DEWP' in df.columns and 'TEMP' in df.columns: # dew point cannot exceed air temp
        bad = df['DEWP'] > (df['TEMP'] + 2)
        impossible += bad.sum()
        df.loc[bad, 'DEWP'] = np.nan
    if impossible > 0:
        print(f'  2.4 | removed {impossible} physically impossible values')

    # ── 2.5 Tier 2: flag real extreme events (IMD thresholds) ────────────
    # these stay in the data — they are the signal, not noise
    if 'PRCP' in df.columns:
        df['HEAVY_RAIN_DAY']      = df['PRCP'] >= IMD_HEAVY_RAIN
        df['VERY_HEAVY_RAIN_DAY'] = df['PRCP'] >= IMD_VERY_HEAVY
        df['EXTREME_RAIN_DAY']    = df['PRCP'] >= IMD_EXTREME_RAIN
        print(f'  2.5 Tier2 | flagged {int(df["HEAVY_RAIN_DAY"].sum())} '
              f'heavy rain days (>={IMD_HEAVY_RAIN}mm) — kept')
    if 'MAX' in df.columns:
        df['HEATWAVE_DAY']    = df['MAX'] >= IMD_HEATWAVE
        df['SEVERE_HEAT_DAY'] = df['MAX'] >= IMD_SEVERE_HEAT
        print(f'  2.5 Tier2 | flagged {int(df["HEATWAVE_DAY"].sum())} '
              f'heatwave days (MAX>={IMD_HEATWAVE}C) — kept')

    # ── 2.5 Tier 2: mark known historical extreme events ─────────────────
    df['KNOWN_EXTREME']      = False
    df['KNOWN_EXTREME_DESC'] = ''
    for (ev_city, ev_year, ev_month, ev_var, ev_desc) in KNOWN_EXTREMES:
        if ev_city == city_name:
            mask = (df['YEAR'] == ev_year) & (df['MONTH'] == ev_month)
            df.loc[mask, 'KNOWN_EXTREME']      = True
            df.loc[mask, 'KNOWN_EXTREME_DESC'] = ev_desc
    known_count = int(df['KNOWN_EXTREME'].sum())
    if known_count > 0:
        print(f'  2.5 Tier2 | marked {known_count} rows as known historical extremes')

    # ── 2.5 Tier 3: robust anomaly scoring (median + MAD) ─────────────────
    # NOT mean + SD — a real flood inflates the mean, making the flood look
    # less extreme than it is. Median and MAD avoid this problem.
    analysis_vars = ['TEMP', 'MAX', 'MIN', 'PRCP', 'WDSP', 'DEWP']
    for col in analysis_vars:
        if col not in df.columns:
            continue
        monthly_median = df.groupby('MONTH')[col].transform('median')
        monthly_mad    = df.groupby('MONTH')[col].transform(
            lambda x: (x - x.median()).abs().median()
        )
        df[col + '_zscore']  = np.where(
            monthly_mad > 0,
            0.6745 * (df[col] - monthly_median) / monthly_mad,
            0.0
        )
        df[col + '_anomaly'] = df[col + '_zscore'].abs() > 3.5
        # known extreme events override: they are real, not errors
        df.loc[df['KNOWN_EXTREME'] == True, col + '_anomaly'] = False
        df[col + '_pctile']  = df.groupby('MONTH')[col].rank(pct=True).round(3)
    print(f'  2.5 Tier3 | robust z-scores and anomaly flags computed')

    # drop DATE — only YEAR and MONTH needed going forward
    df = df.drop(columns=['DATE'], errors='ignore')
    rows_end = len(df)
    print(f'  done — {rows_end:,} clean rows  '
          f'({rows_start - rows_end} removed, {rows_end} kept)')
    return df

print('Cleaning function defined')

## 2.7 — Run Cleaning on All Four Cities

In [ ]:
cleaned_data = {}
for city, df_raw in raw_data.items():
    if df_raw is not None:
        cleaned_data[city] = clean_city_data(df_raw, city)

all_cities = pd.concat(cleaned_data.values(), ignore_index=True)
print(f'\nCombined cleaned dataset: {len(all_cities):,} rows, {all_cities.shape[1]} columns')
print('\nFinal column list:')
print([c for c in all_cities.columns])

## 2.8 — Confirm Known Extreme Events Are Present

Quick check to confirm that flagged historical events survived the cleaning pipeline and are correctly labelled.

In [ ]:
print('=' * 65)
print('KNOWN HISTORICAL EXTREME EVENTS — PRESENT IN CLEANED DATA')
print('=' * 65)

for city, df in cleaned_data.items():
    known = df[df['KNOWN_EXTREME'] == True].copy()
    if len(known) == 0:
        continue
    print(f'\n{city} — {len(known)} rows flagged')
    print('-' * 55)
    cols = ['YEAR', 'MONTH', 'KNOWN_EXTREME_DESC']
    if 'PRCP' in known.columns: cols.append('PRCP')
    if 'MAX'  in known.columns: cols.append('MAX')
    display_df = (
        known[cols]
        .drop_duplicates(subset=['YEAR', 'MONTH'])
        .rename(columns={'PRCP': 'PRCP_mm', 'MAX': 'MAX_C'})
    )
    print(display_df.to_string(index=False))

# confirm Mumbai 2005 flood is at the top of rainfall rankings
if 'Mumbai' in cleaned_data:
    print('\nMumbai — top 5 highest precipitation days:')
    top = cleaned_data['Mumbai'].nlargest(5, 'PRCP')[
        ['YEAR', 'MONTH', 'PRCP', 'KNOWN_EXTREME', 'KNOWN_EXTREME_DESC']
    ]
    print(top.to_string(index=False))

## 2.9 — Store All Data in Memory

Instead of saving to CSV files, all cleaned data is stored in two in-memory structures that can be called anywhere in the notebook:

- `cleaned_data` — a **dictionary** keyed by city name. Access one city with `cleaned_data['Mumbai']`
- `all_cities` — a single **combined dataframe** of all four cities together. Filter by city using `all_cities[all_cities['CITY'] == 'Delhi']`
- `city_list` — a plain **list** of city names for looping
- `data_list` — a **list of dataframes** in the same order as `city_list`

In [ ]:
# dictionary — access by city name
# cleaned_data['Mumbai']  -> Mumbai dataframe
# cleaned_data['Delhi']   -> Delhi dataframe
# cleaned_data['Dehradun'] -> Dehradun dataframe
# cleaned_data['Jodhpur'] -> Jodhpur dataframe
# already built in cell 2.7 — just confirming it is accessible here

# combined dataframe of all four cities
# already built in cell 2.7 — confirming accessible
# filter by city:  all_cities[all_cities['CITY'] == 'Mumbai']
# filter by season: all_cities[all_cities['SEASON'] == 'Monsoon']

# list versions — useful for looping
city_list = list(cleaned_data.keys())
data_list = list(cleaned_data.values())

# quick access examples
mumbai   = cleaned_data['Mumbai']
delhi    = cleaned_data['Delhi']
dehradun = cleaned_data['Dehradun']
jodhpur  = cleaned_data['Jodhpur']

print('Data stored in memory — access methods:')
print()
print('  cleaned_data[city]          -> one city dataframe')
print('  all_cities                  -> all four cities combined')
print('  city_list                   -> list of city names')
print('  data_list                   -> list of dataframes')
print('  mumbai / delhi / dehradun / jodhpur  -> direct variables')
print()
print('Shapes:')
for city in city_list:
    df = cleaned_data[city]
    print(f'  {city:10s}  {df.shape[0]:,} rows  x  {df.shape[1]} columns')
print(f'  {"all_cities":10s}  {all_cities.shape[0]:,} rows  x  {all_cities.shape[1]} columns')
print()
print('Example calls:')
print(f'  mumbai.head(2)')
print(f'  cleaned_data["Delhi"]["TEMP"].mean()')
print(f'  all_cities[all_cities["CITY"] == "Jodhpur"]["MAX"].max()')

data points for all cities are store as list u can cal each thing by calling that and than iterating ver years and months .
Done till part 2(Data Downloade &Cleaned )